# H05 — 给工具接上外部设备

**学习入口：把下面 prompt 发给 VS Code Codex 侧边栏。** Notebook 也可自行逐格阅读。
C++ 在持续 ESP-IDF 工程编辑与构建；Notebook 运行仅显示示例/手动证据卡。
**状态：Alpha；固件构建与实机验收分开记录，见 ../VALIDATION.md。**

```text
开始 H05 — 给工具接上外部设备。
请先读取本工程 AGENTS.md、.academy/course.json 和 .academy/progress.md。
通过 course_root 找到 step2/course.json 和 step2/notebooks/H05-buses.ipynb。
读取 Notebook 的 cells/source 与 academy.step_id/mode；先读当前步骤相关格。
检查工程、环境和前置条件，从第一个未完成步骤开始；每轮只推进一个小步骤。
Guided 可合并指定示例并说明 diff；Modify/Debug/Build/Open 先让我尝试，按需逐级提示。
不要主动读取 solution.md 或其他答案。环境故障可以排查修复，解释后回到原步骤。
不要把挑战 Bug 自动修掉。以当前代码与真实证据验收，记录提示级别和下一步。
我已有一定 C++ 基础，Python 几乎零基础。现在开始。
```

不用 agent 时按步骤标题顺序操作；卡住后再逐级查看本课 hints.md。启动 prompt 的路径按工作区课程链接解析，不靠其他窗口或聊天记忆。

## 我们要解决什么问题？

外设不是“调用一次 API 就有数据”。本单元拆成 UART、I2C、SPI 三个小实验，一次只接一个；把超时、半帧和事务语义留在驱动层。

前置：H04；UART/SPI 只需跳线，BH1750 为 I2C 实验所需。

## 本次够用的知识

UART 是字节流，读取块不等于完整消息；I2C 按地址访问，有 ACK 和上拉；SPI 是同步事务，CS、mode、频率和长度单位都属于设备契约。选择接口要考虑设备支持、距离、速率、引脚与故障处理。

先预测、运行和记录，再补概念。每节开始保存现有工程差异，后续沿用同一工程。

## 可选：Notebook 示例与证据卡

使用课程 .venv kernel 运行下格；这只是显示工具，不需要学习 Python。侧边栏学习不依赖这些格，build/flash/monitor 都在 ESP-IDF Terminal 执行。

In [ ]:
from pathlib import Path
import sys
import json

course_root = None
for candidate in [Path.cwd(), *Path.cwd().parents]:
    if (candidate / "step2/course.json").is_file():
        course_root = candidate
        break
    link = candidate / ".academy/course.json"
    if link.is_file():
        course_root = Path(json.loads(link.read_text(encoding="utf-8"))["course_root"])
        break
if course_root is None or not (course_root / "step2/course.json").is_file():
    raise RuntimeError("请从课程工作区运行，或先在学习工程 .academy/course.json 链接课程根目录")
if str(course_root) not in sys.path:
    sys.path.insert(0, str(course_root))
from tools.esp32_notebook import install, hardware_checklist
install(get_ipython())


## 分开实验并核对电平 · Guided

读 ../BOARD.md 的总线分配。UART 回环只短接本板 GPIO17(TX2) 与 GPIO16(RX2)，不接其他设备；ESP32-WROVER 可能占用这些脚，必须另选可用 GPIO。
I2C 选 3.3 V 可用的 BH1750 模块，SDA21/SCL22，确认模块上拉接 3.3 V；SPI 回环断开所有外设后接 MOSI13→MISO32，不用 GPIO12。每次接线先断电。

**停在这里：** 完成本步骤后记录预期、实际和证据，再进入下一步；agent 等待你的结果。

## 一块数据不等于一帧 · Guided

REQUIRES 增加 esp_driver_uart。跑 UART2 回环，记录 sent、chunk、timeout；monitor 仍使用 UART0/USB，不与练习 UART2 混用。
本示例只观察 chunk。拆开回环再恢复，确认不会永久阻塞，之后再给 
 分隔帧加 parser。

**停在这里：** 完成本步骤后记录预期、实际和证据，再进入下一步；agent 等待你的结果。

示例来源：`step2/lessons/H05/examples/uart_loopback.cpp`。目标：`main/main.cpp`。这是教学示例；合并后以工程源码为准，后续不要来回修改两份实现。依赖参考：`REQUIRES esp_driver_uart esp_driver_i2c esp_driver_spi freertos log`；SRCS 按本步骤目标文件登记。

In [ ]:
%%esp32_example --target main/main.cpp
#include "driver/uart.h"
#include "esp_log.h"
#include "freertos/FreeRTOS.h"
#include "freertos/task.h"

extern "C" void app_main() {
    uart_config_t cfg{};
    cfg.baud_rate = 115200;
    cfg.data_bits = UART_DATA_8_BITS;
    cfg.parity = UART_PARITY_DISABLE;
    cfg.stop_bits = UART_STOP_BITS_1;
    cfg.flow_ctrl = UART_HW_FLOWCTRL_DISABLE;
    cfg.source_clk = UART_SCLK_DEFAULT;
    ESP_ERROR_CHECK(uart_param_config(UART_NUM_2, &cfg));
    ESP_ERROR_CHECK(
        uart_set_pin(UART_NUM_2, 17, 16, UART_PIN_NO_CHANGE, UART_PIN_NO_CHANGE));
    ESP_ERROR_CHECK(uart_driver_install(UART_NUM_2, 256, 0, 0, nullptr, 0));
    const char message[] = "STATUS\n";
    uint8_t data[32];
    while (true) {
        const int sent = uart_write_bytes(UART_NUM_2, message, sizeof(message) - 1);
        const int received =
            uart_read_bytes(UART_NUM_2, data, sizeof(data), pdMS_TO_TICKS(200));
        if (received > 0) {
            ESP_LOGI("uart", "sent=%d chunk=%d data=%.*s", sent, received, received,
                     data);
        } else {
            ESP_LOGW("uart", "sent=%d no data (timeout/error=%d)", sent, received);
        }
        vTaskDelay(pdMS_TO_TICKS(1000));
    }
}


## 传感器会不回答 · Guided

REQUIRES 增加 esp_driver_i2c，使用 IDF 新 i2c_master API，不能和旧 driver/i2c.h 混装同一外设。
BH1750 ADDR low 为 0x23，0x10 连续高分辨率模式后等待至少一个转换周期，读两字节高位在前。把错误与上一次 lux 分开显示；lux_estimate 仍受器件误差影响。

**停在这里：** 完成本步骤后记录预期、实际和证据，再进入下一步；agent 等待你的结果。

示例来源：`step2/lessons/H05/examples/i2c_light.cpp`。目标：`main/main.cpp`。这是教学示例；合并后以工程源码为准，后续不要来回修改两份实现。依赖参考：`REQUIRES esp_driver_uart esp_driver_i2c esp_driver_spi freertos log`；SRCS 按本步骤目标文件登记。

In [ ]:
%%esp32_example --target main/main.cpp
#include "driver/i2c_master.h"
#include "esp_log.h"
#include "freertos/FreeRTOS.h"
#include "freertos/task.h"

extern "C" void app_main() {
    i2c_master_bus_config_t bus_cfg{};
    bus_cfg.i2c_port = I2C_NUM_0;
    bus_cfg.sda_io_num = GPIO_NUM_21;
    bus_cfg.scl_io_num = GPIO_NUM_22;
    bus_cfg.clk_source = I2C_CLK_SRC_DEFAULT;
    bus_cfg.glitch_ignore_cnt = 7;
    // Module/external pull-ups to 3.3 V are required; internal pull-ups are not a bus design.
    i2c_master_bus_handle_t bus = nullptr;
    ESP_ERROR_CHECK(i2c_new_master_bus(&bus_cfg, &bus));
    i2c_device_config_t device_cfg{};
    device_cfg.dev_addr_length = I2C_ADDR_BIT_LEN_7;
    device_cfg.device_address = 0x23;
    device_cfg.scl_speed_hz = 100000;
    i2c_master_dev_handle_t sensor = nullptr;
    ESP_ERROR_CHECK(i2c_master_bus_add_device(bus, &device_cfg, &sensor));
    while (true) {
        // BH1750 continuous high-resolution mode. ADDR low -> 0x23.
        const uint8_t command = 0x10;
        esp_err_t err = i2c_master_transmit(sensor, &command, 1, 100);
        if (err == ESP_OK) {
            vTaskDelay(pdMS_TO_TICKS(180));
            uint8_t bytes[2]{};
            err = i2c_master_receive(sensor, bytes, 2, 100);
            if (err == ESP_OK) {
                const unsigned raw = (unsigned(bytes[0]) << 8) | bytes[1];
                ESP_LOGI("i2c", "BH1750 raw=%u lux_estimate=%.1f", raw, raw / 1.2);
            }
        }
        if (err != ESP_OK) {
            ESP_LOGW("i2c", "transaction failed: %s; retry in 1s",
                     esp_err_to_name(err));
        }
        vTaskDelay(pdMS_TO_TICKS(1000));
    }
}


## 观察一笔同步事务 · Guided

REQUIRES 增加 esp_driver_spi。先跑无外设 SPI 回环，输出四字节一致才通过；示例不是显示屏驱动。
记录 mode=0、1 MHz、length 单位 bits 与 CS27。若选屏幕扩展，先按型号资料实现复位、初始化、DC/CS、像素格式，不把回环 PASS 当作屏幕亮。

**停在这里：** 完成本步骤后记录预期、实际和证据，再进入下一步；agent 等待你的结果。

示例来源：`step2/lessons/H05/examples/spi_loopback.cpp`。目标：`main/main.cpp`。这是教学示例；合并后以工程源码为准，后续不要来回修改两份实现。依赖参考：`REQUIRES esp_driver_uart esp_driver_i2c esp_driver_spi freertos log`；SRCS 按本步骤目标文件登记。

In [ ]:
%%esp32_example --target main/main.cpp
#include <cstring>
#include "driver/spi_master.h"
#include "esp_log.h"
#include "freertos/FreeRTOS.h"
#include "freertos/task.h"

extern "C" void app_main() {
    // Disconnect any external SPI device; connect GPIO13(MOSI) to GPIO32(MISO).
    spi_bus_config_t bus{};
    bus.mosi_io_num = 13;
    bus.miso_io_num = 32;
    bus.sclk_io_num = 14;
    bus.quadwp_io_num = -1;
    bus.quadhd_io_num = -1;
    ESP_ERROR_CHECK(spi_bus_initialize(SPI2_HOST, &bus, SPI_DMA_DISABLED));
    spi_device_interface_config_t cfg{};
    cfg.clock_speed_hz = 1000000;
    cfg.mode = 0;
    cfg.spics_io_num = 27;
    cfg.queue_size = 1;
    spi_device_handle_t device = nullptr;
    ESP_ERROR_CHECK(spi_bus_add_device(SPI2_HOST, &cfg, &device));
    uint8_t tx[] = {0x12, 0x34, 0x56, 0x78};
    uint8_t rx[sizeof(tx)]{};
    spi_transaction_t transfer{};
    transfer.length = sizeof(tx) * 8; // bits, not bytes
    transfer.tx_buffer = tx;
    transfer.rx_buffer = rx;
    ESP_ERROR_CHECK(spi_device_transmit(device, &transfer));
    ESP_LOGI("spi", "loopback=%s",
             std::memcmp(tx, rx, sizeof(tx)) == 0 ? "PASS" : "FAIL");
    while (true) {
        vTaskDelay(pdMS_TO_TICKS(1000));
    }
}


## 换一种输入节奏 · Modify

UART 分两次发送 STAT 和 US
，中间间隔 50 ms；日志会得到几块？把 parser 输入切成每字节、半帧、粘帧。
I2C 改采样间隔，验证等待转换时间不能跟着随意缩短。SPI 降频后比较；不要同时改接线和软件参数，保留单变量实验。

**停在这里：** 完成本步骤后记录预期、实际和证据，再进入下一步；agent 等待你的结果。

## 长度与地址 · Debug

软件将 BH1750 地址改为 0x24，观察具体错误并恢复；SPI transfer.length 暂时从 32 改成 4，分析只传四 bit 的证据。
不要拔线时同时短接信号，也不要把所有错误都转成零读数。断电拆传感器后重启，检查失败能传播到业务层。

**停在这里：** 完成本步骤后记录预期、实际和证据，再进入下一步；agent 等待你的结果。

## 一个有边界的驱动 · Build

实现 UART 固定容量行 parser：半帧缓存、粘帧分割、溢出丢弃到分隔符、timeout 清理；错误不是正常消息。
实现 BH1750 驱动 read 返回有效值或具体 error，业务层负责 degraded/retry；用 host fake 测试 NACK、timeout 和恢复。SPI 显示是选修，先写设备接口再选受维护驱动。

**停在这里：** 完成本步骤后记录预期、实际和证据，再进入下一步；agent 等待你的结果。

可选自动验收：按 `step2/tests/README.md` 的接口契约，显式指定自己的 header 路径运行 CMake/CTest。默认测维护者参考，不能计作自己的代码通过。


## 选择你真正需要的外设 · Open

为最终工具选择一种通信接口，列出兼容设备、pin budget、错误模型与恢复策略。恢复 H03/H04 的按钮响应要求，测传感器断线时按钮是否仍可用。
H06 再根据已出现的代码复杂度拆组件。

**停在这里：** 完成本步骤后记录预期、实际和证据，再进入下一步；agent 等待你的结果。

## 验收与复盘

- [ ] UART 断线超时、半帧和粘帧有证据
- [ ] I2C 地址/转换时间/错误与有效数据分开
- [ ] SPI 回环与显示器功能分开验收

分别记录 host、firmware build、flash、日志、实物与网络往返；未知保持未确认。手动卡片只是学习者记录，不自动测量，也不自动写入进度文件。

复盘：我选择了什么机制？哪个证据排除了错误假设？哪些内容还需要测量？把当前 Step、Hint level、代码版本、证据和下次起点写入 `.academy/progress.md`。

In [ ]:
hardware_checklist(
    'H05',
    [
        'UART 断线超时、半帧和粘帧有证据',
        'I2C 地址/转换时间/错误与有效数据分开',
        'SPI 回环与显示器功能分开验收',
    ],
)


## 遇到困难时

先完成一次尝试，再打开 `step2/lessons/H05/hints.md`，每次只读当前步骤的一层。完整参考与解题分析单独保存，默认路径不主动读取。

## 官方版本参考

- [api-reference/peripherals/uart.html](https://docs.espressif.com/projects/esp-idf/en/v5.5.1/esp32/api-reference/peripherals/uart.html)
- [api-reference/peripherals/i2c.html](https://docs.espressif.com/projects/esp-idf/en/v5.5.1/esp32/api-reference/peripherals/i2c.html)
- [api-reference/peripherals/spi_master.html](https://docs.espressif.com/projects/esp-idf/en/v5.5.1/esp32/api-reference/peripherals/spi_master.html)